In [1]:
print(123)

123


In [2]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
print(len(documents))

72


In [5]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [13]:
from pydantic import BaseModel
from evaluation_utils import llm_structured, calc_price, calc_total_price, llm_structured_retry, map_progress, llm_structured
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv('llm.env')
openai_client = OpenAI()

class Questions(BaseModel):
    questions: list[str]

In [14]:
records = []
usages = []

for doc in documents:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"]
    })

    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="gpt-5.4-mini",
    )

    for question in result.questions:
        records.append({
            "question": question,
            "filename": doc["filename"]
        })

    usages.append(usage)


In [16]:
print(documents[0])

{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a simp

In [19]:
documents[71]

{'content': '# Chunking for Longer Texts\n\n<a href="https://www.youtube.com/watch?v=tyBRP_WewXA&list=PL3MmuxUbc_hIB4fSqLy_0AfTjVLpgjV3R">\n  <img src="https://markdown-videos-api.jorgenkh.no/youtube/tyBRP_WewXA">\n</a>\n\nOur FAQ data is well-structured: each document is a question-answer\npair. But what if your data is articles, transcripts, or slide\ndecks? You need to chunk it into pieces that are the right size\nfor embedding and retrieval.\n\n## Multiple articles\n\nIf you have multiple articles (blog posts, wiki pages, etc.\n\n):\n\n1. Assign each article a document ID\n2. Split each article into chunks\n3. Give each chunk a unique chunk ID (e.g., `doc_id_1`, `doc_id_2`)\n4. Evaluate retrieval with separate Hit Rate for both document ID\n   and chunk ID\n5. Tune chunk size using RAG evaluation metrics\n\n```json\n{\n  "doc_id": "abc123",\n  "chunk_id": "abc123_1",\n  "text": "first paragraph of the article..."\n}\n```\n\n## Single article or transcript\n\nIf you have one long pi

In [21]:
records[0], usages[0]

({'question': 'What exactly is RAG, and why does it help when an LLM doesn’t know something or can’t see my data?',
  'filename': '01-agentic-rag/lessons/01-intro.md'},
 ResponseUsage(input_tokens=1021, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=124, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1145))

In [22]:
records = []
usages = []

for doc in documents[:3]:
    user_prompt = json.dumps({
        "filename": doc["filename"],
        "content": doc["content"]
    })

    result, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions,
        model="gpt-5.4-mini",
    )

    for question in result.questions:
        records.append({
            "question": question,
            "filename": doc["filename"]
        })

    usages.append(usage)


In [ ]:
usages

#Average input_tokens = 1354 ~ 1400

[ResponseUsage(input_tokens=1021, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=112, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1133),
 ResponseUsage(input_tokens=1287, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=102, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1389),
 ResponseUsage(input_tokens=1754, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=92, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1846)]